# Matrix Factorization — Matrix Factorization Techniques for Recommender Systems (2009)

_Papers · Recommender Systems_

**Predict a user's rating of an item as the dot product of two learned vectors — one per user, one per item — and fit them by gradient descent on the ratings you have actually seen.**

---

This notebook is a paced, step-by-step walkthrough of **Matrix Factorization — Matrix Factorization Techniques for Recommender Systems (2009)**. We build it one piece at a time: a hand-computed SGD update, factorizing a hidden low-rank matrix from a fraction of its entries, and a check that our hand-derived gradient matches autograd. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — One SGD update, by hand

The model predicts a rating as the **dot product** of a user vector `p_u` and an item vector `q_i` (Eq. 1). Given the error `e_ui = r_ui - pred`, regularized SGD nudges each vector toward reducing that error while a `lam` term pulls it back toward zero. Crucially the `q_i` and `p_u` updates both use the **old** `q_i`, so we compute the predictions and error first, then apply both updates.

In [ ]:
import torch

torch.manual_seed(0)

p_u = torch.tensor([1.0, 2.0])                    # a user factor vector
q_i = torch.tensor([0.5, -1.0])                   # an item factor vector
r_ui, gamma, lam = 3.0, 0.05, 0.1                 # true rating, learning rate, regularization

pred = (q_i * p_u).sum()                          # Eq 1: dot product  -> -1.5
e_ui = r_ui - pred                                # error              ->  4.5
print("pred:", pred.item(), " e_ui:", e_ui.item())

q_new = q_i + gamma * (e_ui * p_u - lam * q_i)    # SGD update for q_i
p_new = p_u + gamma * (e_ui * q_i - lam * p_u)    # SGD update for p_u (uses OLD q_i)
print("q_new:", [round(x, 4) for x in q_new.tolist()])   # [0.7225, -0.545]
print("p_new:", [round(x, 4) for x in p_new.tolist()])   # [1.1075,  1.765]

### Step 2 — Factorize a hidden low-rank matrix from 70% of its cells

We generate a true rank-2 ratings matrix `R = P_true @ Q_true.T`, then hide 30% of its cells and train factors `P`, `Q` only on the **observed** set (the set κ). Each epoch loops over the observed (u, i) pairs in random order and applies the same regularized SGD update from Step 1 — keeping the old `P[u]` for the `Q[i]` step. Because the data is exactly rank-2 and clean, fitting the visible 70% recovers the hidden 30% too, which the train vs held-out RMSE confirms.

In [ ]:
# Generate a TRUE rank-2 ratings matrix.
n_users, n_items, K_true, f = 8, 6, 2, 2
P_true = torch.randn(n_users, K_true)
Q_true = torch.randn(n_items, K_true)
R = P_true @ Q_true.T                             # the TRUE rank-2 ratings matrix

# Observe only 70% of the cells (the set K = kappa).
mask = torch.rand(n_users, n_items) < 0.7
obs  = mask.nonzero(as_tuple=False)               # list of observed (u, i) pairs

# Learnable factors, initialized small.
P = torch.randn(n_users, f) * 0.1
Q = torch.randn(n_items, f) * 0.1
gamma, lam = 0.02, 0.05

for epoch in range(4000):                         # loop over the training set, Eq 2 via SGD
    for u, i in obs[torch.randperm(obs.size(0))]:
        e  = R[u, i] - (P[u] @ Q[i])              # e_ui = r_ui - q_i^T p_u
        Pu = P[u].clone()                         # keep OLD p_u for the q_i update
        P[u] = P[u] + gamma * (e * Q[i] - lam * P[u])
        Q[i] = Q[i] + gamma * (e * Pu  - lam * Q[i])

Rhat = P @ Q.T                                    # predicted full matrix
train_rmse = ((R[mask]  - Rhat[mask])**2).mean().sqrt()
test_rmse  = ((R[~mask] - Rhat[~mask])**2).mean().sqrt()
print("train RMSE (observed):", round(train_rmse.item(), 4))   # 0.0759
print("test  RMSE (held-out):", round(test_rmse.item(),  4))   # 0.1793
print("R   [0,:3]:", [round(x, 3) for x in R[0, :3].tolist()])    # [1.275, 0.462, -0.304]
print("Rhat[0,:3]:", [round(x, 3) for x in Rhat[0, :3].tolist()]) # [1.166, 0.39, -0.295]

### Step 3 — Verify the hand-coded gradient against autograd

Our SGD update came from a hand-derived gradient — so let's prove it's correct. We build the per-cell regularized squared-error loss with autograd-tracked copies of `P` and `Q`, call `loss.backward()`, and compare PyTorch's `dL/dP[0]` to our manual `-2(e·q_i - lam·p_u)`. `torch.allclose` returning True means the rule we coded by hand *is* the true gradient.

In [ ]:
# Autograd-tracked copies for the cell (u=0, i=0).
P2 = P.clone().detach().requires_grad_(True)
Q2 = Q.clone().detach().requires_grad_(True)
u, i = 0, 0

loss = (R[u, i] - (P2[u] @ Q2[i]))**2 + lam * ((P2[u]**2).sum() + (Q2[i]**2).sum())
loss.backward()

manual = -2 * ((R[u, i] - (P[u] @ Q[i])) * Q[i] - lam * P[u])    # d/dP[u] of the per-cell loss
print("autograd dL/dP[0]:", [round(x, 4) for x in P2.grad[0].tolist()])  # [-0.0413, -0.0485]
print("manual -2(eQ-lamP):", [round(x, 4) for x in manual.tolist()])     # [-0.0413, -0.0485]
print("gradient match:", torch.allclose(P2.grad[0], manual, atol=1e-5))  # True
# My hand-derived SGD update IS the autograd gradient -- that's the Track A payoff.

## Visualize the data & results

_Can matrix factorization recover the hidden cells of a low-rank matrix from a fraction of its entries, and what does regularization cost when the data is clean?_

### Step 4 — Wrap the whole fit in a function of lambda

To study what regularization costs we package Step 2's fit into a `run(lam)` function: generate a fresh true rank-2 matrix, observe 70%, train the factors with the regularized SGD, and return train and held-out RMSE. Everything is identical to Step 2 except that `lam` is now a knob we can sweep.

In [ ]:
import torch

# Recover a hidden rank-2 matrix at a given lambda; return train vs held-out RMSE.
def run(lam, gamma=0.02, epochs=4000, seed=0):
    torch.manual_seed(seed)
    nU, nI, Kt, f = 8, 6, 2, 2
    R = torch.randn(nU, Kt) @ torch.randn(nI, Kt).T     # true rank-2 matrix
    mask = torch.rand(nU, nI) < 0.7                     # observe 70%
    obs  = mask.nonzero(as_tuple=False)
    P = torch.randn(nU, f) * 0.1
    Q = torch.randn(nI, f) * 0.1
    for _ in range(epochs):
        for u, i in obs[torch.randperm(obs.size(0))]:
            e  = R[u, i] - (P[u] @ Q[i])
            Pu = P[u].clone()
            P[u] = P[u] + gamma * (e * Q[i] - lam * P[u])
            Q[i] = Q[i] + gamma * (e * Pu  - lam * Q[i])
    Rhat = P @ Q.T
    tr = ((R[mask]  - Rhat[mask])**2).mean().sqrt().item()
    te = ((R[~mask] - Rhat[~mask])**2).mean().sqrt().item()
    return tr, te

### Step 5 — Sweep lambda and read off the trade-off

Sweep `lam` from 0 upward and print train and held-out RMSE at each value. On this clean, exactly rank-2 data the regularizer is pure cost — both errors rise monotonically from a near-perfect fit at λ=0 — because there is no noise for λ to guard against. With noisy real ratings the held-out curve would instead be U-shaped, which is the bias-variance trade-off the λ term controls.

In [ ]:
for lam in [0.0, 0.01, 0.05, 0.1, 0.2, 0.4, 0.8]:
    tr, te = run(lam)
    print(f"lambda={lam:<5} train={tr:.4f}  held-out={te:.4f}")
# lambda=0.0   train=0.0000  held-out=0.0000   <- clean low-rank data: perfect recovery
# lambda=0.01  train=0.0157  held-out=0.0394
# lambda=0.05  train=0.0759  held-out=0.1793
# lambda=0.1   train=0.1474  held-out=0.3303
# lambda=0.2   train=0.2802  held-out=0.5719
# lambda=0.4   train=0.5210  held-out=0.8470
# lambda=0.8   train=0.8475  held-out=0.9393
# Noise-free + exactly rank-2: lambda only hurts. With noisy ratings a moderate
# lambda would instead minimize held-out error (the bias-variance trade-off).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** One SGD step. A user vector is $p_u = [2, 0]$ and an item vector is $q_i = [1, 1]$. The
            true rating is $r_{ui} = 5$, the learning rate is $\gamma = 0.1$, and $\lambda = 0$ (no
            regularization for this exercise). Compute the prediction, the error, and the updated $q_i$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Predict with Equation 1: $\hat{r}_{ui} = q_i^\top p_u = (1)(2) + (1)(0) = 2$. — _The prediction is the dot product of the two vectors._
- Error: $e_{ui} = r_{ui} - \hat{r}_{ui} = 5 - 2 = 3$. — _Positive error means we under-predicted; the update will raise the dot product._
- Update $q_i$ with $\lambda = 0$: $q_i + \gamma\,e_{ui}\,p_u = [1,1] + 0.1\cdot 3\cdot[2,0] = [1,1] + [0.6, 0] = [1.6, 1]$. — _With no regularization the rule is $q_i \leftarrow q_i + \gamma e_{ui} p_u$._

**Answer:** Prediction $\hat{r}_{ui} = 2$, error $e_{ui} = 3$, updated $q_i = [\mathbf{1.6,\ 1}]$. The
                 first entry grew (because $p_{u1}=2$ pulled it up) while the second stayed (because
                 $p_{u2}=0$ contributed nothing), nudging the next prediction toward the true $5$.

</details>

**Problem 2.** Why sum over $\kappa$ only? Suppose instead you treat every unobserved cell as a $0$ rating
            and fit the squared error over the entire matrix. Most cells are empty. What does the model
            learn, and why does the paper insist on summing over the observed set $\kappa$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count where the signal is: in a sparse table the vast majority of cells are unobserved. — _Each user rates only a tiny fraction of items, so empties dominate._
- Treating empties as $0$ makes the dominant objective "predict 0 everywhere," since that is what most cells now demand. — _Squared error over all cells is overwhelmed by the empties, dragging every prediction toward 0._
- Conclude the model would push all dot products toward 0 and ignore the real ratings. — _"Not yet rated" is not the same as "rated zero" &mdash; conflating them destroys the signal._

**Answer:** It would learn to predict near-$0$ for everything, because the empty cells (treated as $0$)
                 swamp the loss. "Unobserved" means "unknown," not "zero." The paper sums over $\kappa$,
                 the observed pairs only, precisely so the model fits real ratings and is free to predict
                 whatever best explains them in the holes. (The paper notes naive imputation of the
                 missing entries is expensive and can distort the data.)

</details>

**Problem 3.** The ablation: what does $\lambda$ cost on clean data? Our notebook recovers a hidden,
            noise-free rank-2 matrix. Predict what happens to the train and held-out RMSE as you sweep
            $\lambda$ from $0$ upward, and contrast that with what you would expect if the observed ratings
            were noisy (as real ratings are). Explain in terms of Equation 2.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- At $\lambda = 0$, Equation 2 only rewards fitting the observed cells. Because the true matrix is exactly rank-2, fitting the visible 70% pins down the factors, so the hidden cells come out right too. — _A rank-2 matrix has few degrees of freedom; clean observations fully determine it._
- As $\lambda$ grows, the penalty $\lambda(\lVert q_i\rVert^2 + \lVert p_u\rVert^2)$ pulls the factors toward zero, away from the true solution, so BOTH train and held-out RMSE rise. — _With no noise to guard against, the regularizer only adds bias (underfitting)._
- Contrast: if the observed ratings carried noise, $\lambda = 0$ would chase that noise and the held-out RMSE would form a U with its minimum at a moderate $\lambda$. — _Regularization earns its keep by trading a little fit for resistance to noise._

**Answer:** On clean, exactly low-rank data the regularizer is pure cost: in our run (CODEVIZ panel)
                 $\lambda = 0$ recovers the hidden cells almost perfectly (held-out RMSE $\approx 0.0000$),
                 and both train and held-out RMSE rise monotonically as $\lambda$ grows ($0.179$ at
                 $\lambda = 0.05$, $0.939$ at $\lambda = 0.8$). The $\lambda$ term biases the factors toward
                 zero with no noise to justify it. With noisy ratings the story flips &mdash; held-out
                 RMSE becomes U-shaped and a moderate $\lambda$ wins. This is the bias-variance trade-off the
                 $\lambda$ term in Equation 2 controls, and why the paper sets $\lambda$ by
                 cross-validation.

</details>